In [ ]:
# Setup & Installation
# =============================================================================
#!pip install -U langsmith langchain-openai langgraph requests
#!pip install -U ipywidgets jupyter

import os
import json
import requests
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langsmith import Client

In [ ]:
# Config
# =============================================================================

BASE_URL = "http://localhost:8000"
ENDPOINT = "/api/chat"
OPENAI_KEY = "meinKey"

# OpenAI
os.environ["OPENAI_API_KEY"] = OPENAI_KEY

# LangSmith Tracing aktivieren
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "meinKey2"
os.environ["LANGSMITH_PROJECT"] = "Test Agent"

DATASET_NAME = "Invariance Test Dataset"
PASS_THRESHOLD = 0.85

In [ ]:
# Test cases
# =============================================================================

TEST_CASES = [
    {
        "question": "Where is the city of Coesfeld located?",
        "expected": "The city of Coesfeld is located in the district of Coesfeld, NRW, Germany.",
    },
    {
        "question": "Where is the district of Coesfeld located?",
        "expected": "The district of Coesfeld is located in the administrative district of Münster, NRW, Germany.",
    },
    {
        "question": "Where does the city of Coesfeld lie?",
        "expected": "The city of Coesfeld lies in the district of Coesfeld, NRW, Germany.",
    },
    {
        "question": "Where does the district of Coesfeld lie?",
        "expected": "The district of Coesfeld lies in the administrative district of Münster, NRW, Germany.",
    },
    {
        "question": "Where is the city of Coesfeld?",
        "expected": "The city of Coesfeld is in the district of Coesfeld, NRW, Germany.",
    },
    {
        "question": "Where is the district of Coesfeld?",
        "expected": "The district of Coesfeld is in the administrative district of Münster, NRW, Germany.",
    },
]

In [ ]:
# API Call (backend)
# =============================================================================

def chat(message: str, api_key: str) -> str:
    response = requests.post(
        f"{BASE_URL}{ENDPOINT}",
        json={"message": message, "openAiKey": api_key},
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["result"]["result"]

In [ ]:
# AGENT STATE
# =============================================================================

class TestAgentState(TypedDict):
    question: str
    actual: str
    error: str


In [ ]:
# AGENT NODES
# =============================================================================

# NODE 1: INTERPRET INPUT
def interpret_test_case(state: TestAgentState):
    #print("\n====================================")
    #print("Running test")
    #print("Question:", state["question"])
    #print("====================================")
    return state


# NODE 2: RUN API TEST
def run_api_test(state: TestAgentState):
    try:
        actual = chat(state["question"], OPENAI_KEY)
        return {**state, "actual": actual, "error": ""}
    except Exception as e:
        return {**state, "actual": "", "error": str(e)}


# NODE 3: OUTPUT (lokales Debugging - LangSmith zeigt eh alles in der UI)
def output_result(state: TestAgentState):
    #print("Actual :", state["actual"])
    #if state["error"]:
    #    print("Error  :", state["error"])
    #print()
    return state

In [ ]:
# Build the LangGraph workflow
# =============================================================================

workflow = StateGraph(TestAgentState)

workflow.add_node("interpret", interpret_test_case)
workflow.add_node("run_test", run_api_test)
workflow.add_node("output", output_result)

workflow.add_edge(START, "interpret")
workflow.add_edge("interpret", "run_test")
workflow.add_edge("run_test", "output")
workflow.add_edge("output", END)

compiled_graph = workflow.compile()


# Optional: visualisiation
#from IPython.display import Image, display
#try:
#    display(Image(compiled_graph.get_graph().draw_mermaid_png()))
#except Exception:
#    pass

In [ ]:
# LangSmith: build dataset (only at first execution)
# =============================================================================

client = Client()

if not client.has_dataset(dataset_name=DATASET_NAME):
    print(f"Creating dataset '{DATASET_NAME}' in LangSmith...")
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Coesfeld location questions for chat backend testing",
    )
    client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {
                "inputs": {"question": tc["question"]},
                "outputs": {"expected": tc["expected"]},
            }
            for tc in TEST_CASES
        ],
    )
    print(f"Dataset created with {len(TEST_CASES)} examples.")
else:
    print(f"Dataset '{DATASET_NAME}' already exists - reusing it.")

In [ ]:
# LangSmith: Target function (calls agent)
# =============================================================================

def target(inputs: dict) -> dict:
    """
    Called by LangSmith for each dataset example.
    Runs the entire LangGraph so that all nodes
    appear as trace steps in the LangSmith UI.
    """
    result = compiled_graph.invoke({
        "question": inputs["question"],
        "actual": "",
        "error": "",
    })
    return {
        "actual": result["actual"],
        "error": result["error"],
    }

In [ ]:
# LangSmith: Evaluators (Judge LLM)
# =============================================================================

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

JUDGE_INSTRUCTIONS = """
Given an ACTUAL answer and an EXPECTED answer, determine whether
the actual answer contains all of the information in the expected answer.

Return ONLY valid JSON in this format:
{"score": number between 0 and 1, "reason": "short explanation"}
"""


def _judge(actual: str, expected: str) -> dict:
    """Help function: calls the Judge LLM."""
    if not actual:
        return {"score": 0.0, "reason": "No actual answer (API error)."}

    user_msg = f"""ACTUAL ANSWER:
{actual}

EXPECTED ANSWER:
{expected}
"""
    response = judge_llm.invoke([
        {"role": "system", "content": JUDGE_INSTRUCTIONS},
        {"role": "user", "content": user_msg},
    ])
    try:
        parsed = json.loads(response.content.strip())
        return {"score": float(parsed["score"]), "reason": parsed["reason"]}
    except Exception as e:
        return {"score": 0.0, "reason": f"JSON parsing failed: {e}"}


def semantic_correctness(outputs: dict, reference_outputs: dict) -> dict:
    """LangSmith-Evaluator: semantic similarity (0-1)."""
    judgement = _judge(outputs.get("actual", ""), reference_outputs.get("expected", ""))
    return {
        "key": "semantic_correctness",
        "score": round(judgement["score"], 3),
        "comment": judgement["reason"],
    }


def passed(outputs: dict, reference_outputs: dict) -> dict:
    """LangSmith-Evaluator: Binary PASS/FAIL for simple filtering in the UI."""
    judgement = _judge(outputs.get("actual", ""), reference_outputs.get("expected", ""))
    return {
        "key": "passed",
        "score": 1 if judgement["score"] >= PASS_THRESHOLD else 0,
        "comment": judgement["reason"],
    }


def no_error(outputs: dict, reference_outputs: dict) -> dict:
    """LangSmith-Evaluator: checks whether the API responded at all."""
    return {
        "key": "no_error",
        "score": 0 if outputs.get("error") else 1,
        "comment": outputs.get("error") or "ok",
    }

In [ ]:
# Run experiment in LangSmith
# =============================================================================

import statistics


def _get_attr(obj, name, default=None):
    """Handle both Pydantic objects and dictionaries."""
    if hasattr(obj, name):
        return getattr(obj, name, default)
    if isinstance(obj, dict):
        return obj.get(name, default)
    return default


def run_experiment():
    print(f"\n=== Starting LangSmith Experiment for {len(TEST_CASES)} Tests ===\n")

    experiment_results = client.evaluate(
        target,
        data=DATASET_NAME,
        evaluators=[semantic_correctness, passed, no_error],
        experiment_prefix="coesfeld-test",
        max_concurrency=2,
        metadata={
            "model": "gpt-4o-mini",
            "endpoint": ENDPOINT,
            "base_url": BASE_URL,
            "version": "v1",
        },
    )

    
    # 1) collect results
    # -------------------------------------------------------------------------
    rows = []

    for r in experiment_results:
        example = r["example"]
        run = r["run"]
        eval_objs = r["evaluation_results"]["results"]

        scores = {}
        comments = {}
        for e in eval_objs:
            key = _get_attr(e, "key")
            scores[key] = _get_attr(e, "score", 0)
            comments[key] = _get_attr(e, "comment", "")

        question = example.inputs.get("question", "?")
        expected = (example.outputs or {}).get("expected", "")
        actual = (_get_attr(run, "outputs") or {}).get("actual", "")
        error = (_get_attr(run, "outputs") or {}).get("error", "")

        rows.append({
            "question": question,
            "expected": expected,
            "actual": actual,
            "error": error,
            "semantic_score": scores.get("semantic_correctness", 0),
            "passed": scores.get("passed", 0),
            "no_error": scores.get("no_error", 1),
            "reason": comments.get("semantic_correctness", ""),
        })

    
    # 2) Detailed results per test
    # -------------------------------------------------------------------------
    print("\n" + "=" * 80)
    print("DETAILED RESULTS")
    print("=" * 80)

    for i, row in enumerate(rows, 1):
        print(f"\n[{i}/{len(rows)}] Score: {row['semantic_score']:.2f}")
        print(f"  Q: {row['question']}")
        print(f"  Expected: {row['expected']}")
        print(f"  Actual  : {row['actual'] or '(no answer)'}")
        if row["error"]:
            print(f"  Error   : {row['error']}")
        print(f"  Judge   : {row['reason']}")

    
    # 3) Summary
    # -------------------------------------------------------------------------
    total = len(rows)
    passed_count = sum(1 for r in rows if r["passed"] == 1)
    failed_count = total - passed_count
    error_count = sum(1 for r in rows if r["no_error"] == 0)

    sem_scores = [r["semantic_score"] for r in rows]
    avg_score = statistics.mean(sem_scores) if sem_scores else 0
    median_score = statistics.median(sem_scores) if sem_scores else 0
    min_score = min(sem_scores) if sem_scores else 0
    max_score = max(sem_scores) if sem_scores else 0

    pass_rate = (passed_count / total * 100) if total else 0

    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"  Total tests   : {total}")
    print(f"  Passed     : {passed_count}  ({pass_rate:.1f}%)")
    print(f"  Failed     : {failed_count}")
    print(f"  API errors : {error_count}")
    print("-" * 80)
    print(f"  Semantic score (0-1):")
    print(f"    Average : {avg_score:.3f}")
    print(f"    Median  : {median_score:.3f}")
    print(f"    Min/Max : {min_score:.3f} / {max_score:.3f}")
    print("=" * 80)

    # List of failed questions for a quick overview
    if failed_count > 0:
        print("\nFAILED QUESTIONS:")
        for row in rows:
            if row["passed"] == 0:
                print(f"  - [{row['semantic_score']:.2f}] {row['question']}")
        print("=" * 80)
    
    return experiment_results, rows

In [ ]:
# Main
# =============================================================================

#if __name__ == "__main__":
#    results, rows = run_experiment()
run_experiment()